# SWAP-Stress: AMSR VOD (Vegetation Optical Depth)

This notebook demonstrates the **AMSR-E/AMSR2 LPDR v3** Vegetation Optical Depth (VOD)
data used in SWAP-Stress.

VOD at 10.7 GHz is a microwave-derived proxy for vegetation water content and biomass.
The LPDR v3 product provides daily global coverage on a ~25 km EASE-Grid (EPSG:3410)
from 2002 through 2022, with separate ascending and descending orbit passes.

In the training pipeline, `map.data.amsr_extract` samples the nearest grid cell at each
training site and computes multi-year seasonal statistics (mean, stddev) for 5 periods
(winter, spring, summer, autumn, annual) × 2 passes = **20 features** per site.

This notebook covers:
1. Inspecting a single AMSR NetCDF file
2. Extracting a daily VOD time series at a ReESH site (US-CDM)
3. Comparing seasonal climatology to the pre-computed parquet
4. Summarizing the full climatology parquet
5. Feature distributions across all training sites
6. Spatial pattern of summer VOD across CONUS

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import xarray as xr
except Exception:
    xr = None

try:
    import seaborn as sns
except Exception:
    sns = None

try:
    from pyproj import Transformer
except Exception:
    Transformer = None

# -----------------------------
# Config
# -----------------------------
DATA_ROOT = os.environ.get(
    "SWAPSTRESS_DATA_ROOT",
    os.path.expanduser("~/data/IrrigationGIS/soils"),
)
DATA_ROOT = str(Path(DATA_ROOT).expanduser())

AMSR_DIR = os.environ.get(
    "SWAPSTRESS_AMSR_DIR",
    os.path.join(DATA_ROOT, "amsr"),
)

SITES_PARQUET = os.environ.get(
    "SWAPSTRESS_OBS_TABLE",
    os.path.join(
        DATA_ROOT, "swapstress", "training", "obs_level_training_250m.parquet"
    ),
)

VOD_CLIM_PATH = os.path.join(DATA_ROOT, "amsr", "amsr_vod_climatology.parquet")

SITE_ID = "US-CDM"

OUT_DIR = Path("notebooks/_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("AMSR_DIR:", AMSR_DIR)
print("SITES_PARQUET:", SITES_PARQUET)
print("VOD_CLIM_PATH:", VOD_CLIM_PATH)
print("SITE_ID:", SITE_ID)

## 1) Inspect a single AMSR file

Each NetCDF file covers one year and one orbit pass.  
File naming: `AMSR-E-2_LPDRv3_Y{year}_{A,D}.nc4`  
- `A` = ascending (afternoon equator crossing)  
- `D` = descending (early morning)  

We open one file to see dimensions, coordinate ranges, and VOD valid values.

In [ ]:
assert xr is not None, "xarray is required for this notebook"

# Pick a representative file
sample_file = os.path.join(AMSR_DIR, "AMSR-E-2_LPDRv3_Y2015_A.nc4")
assert os.path.exists(sample_file), f"Missing: {sample_file}"

ds = xr.open_dataset(sample_file)
print("Dimensions:", dict(ds.dims))
print("Variables:", list(ds.data_vars))
print()

# Coordinate ranges (EASE-Grid meters)
print(f"x range: {ds.x.values.min():.0f} to {ds.x.values.max():.0f} m")
print(f"y range: {ds.y.values.min():.0f} to {ds.y.values.max():.0f} m")
print(
    f"time range: {pd.Timestamp(ds.time.values[0])} to {pd.Timestamp(ds.time.values[-1])}"
)
print(f"time steps: {len(ds.time)}")
print()

# VOD stats for one timestep
vod_slice = ds["VOD"].isel(time=180).values
valid = vod_slice[(vod_slice >= 0) & (vod_slice <= 3)]
print(
    f"VOD sample (day 180): shape={vod_slice.shape}, valid pixels={len(valid)}, "
    f"range=[{valid.min():.3f}, {valid.max():.3f}], median={np.median(valid):.3f}"
)

if hasattr(ds["VOD"], "attrs"):
    print("\nVOD attributes:", dict(ds["VOD"].attrs))

ds.close()

## 2) Daily VOD time series at US-CDM

We extract the full daily VOD record (2002–2022) at the ReESH station **US-CDM**
by reprojecting the station coordinates to EASE-Grid and reading the nearest pixel
from each annual file.

In [ ]:
from map.data.amsr_extract import _reproject_sites, _find_nearest_indices

# --- Locate US-CDM in the training parquet ---
sites_df = pd.read_parquet(SITES_PARQUET)
if sites_df.index.name:
    sites_df = sites_df.reset_index()

cdm = sites_df[sites_df["sample_id"].str.startswith(f"reesh_{SITE_ID}")]
site_lat = cdm["lat"].iloc[0]
site_lon = cdm["lon"].iloc[0]
print(f"{SITE_ID}: lat={site_lat:.4f}, lon={site_lon:.4f}")

# --- Reproject to EASE-Grid ---
site_x, site_y = _reproject_sites(np.array([site_lat]), np.array([site_lon]))
print(f"EASE-Grid: x={site_x[0]:.0f}, y={site_y[0]:.0f} m")

# --- Extract daily VOD across all years ---
records = []  # (date, vod_asc, vod_desc)
pass_codes = {"A": "asc", "D": "desc"}
grid_xi = None
grid_yi = None

for year in range(2002, 2023):
    for code, pass_name in pass_codes.items():
        fname = f"AMSR-E-2_LPDRv3_Y{year}_{code}.nc4"
        fpath = os.path.join(AMSR_DIR, fname)
        if not os.path.exists(fpath):
            continue

        ds = xr.open_dataset(fpath)

        # Build grid indices once
        if grid_xi is None:
            gx = ds.x.values
            gy = ds.y.values
            y_flipped = gy[0] > gy[-1]
            if y_flipped:
                gy_sorted = gy[::-1]
            else:
                gy_sorted = gy
            grid_xi = _find_nearest_indices(gx, site_x)[0]
            yi_raw = _find_nearest_indices(gy_sorted, site_y)[0]
            grid_yi = (len(gy) - 1 - yi_raw) if y_flipped else yi_raw

        vod = ds["VOD"].values[:, grid_yi, grid_xi]  # (n_time,)
        vod[(vod < 0) | (vod > 3)] = np.nan
        times = pd.DatetimeIndex(ds.time.values)

        for t in range(len(times)):
            records.append((times[t], pass_name, float(vod[t])))

        ds.close()

ts = pd.DataFrame(records, columns=["date", "pass", "vod"])
ts_wide = ts.pivot_table(index="date", columns="pass", values="vod")
print(
    f"\nDaily records: {len(ts_wide)}, date range: {ts_wide.index.min()} to {ts_wide.index.max()}"
)
print(
    f"Valid asc: {ts_wide['asc'].notna().sum()}, desc: {ts_wide['desc'].notna().sum()}"
)

# --- Plot ---
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, pass_name, color in zip(axes, ["asc", "desc"], ["#2196F3", "#FF9800"]):
    vals = ts_wide[pass_name].dropna()
    ax.scatter(vals.index, vals.values, s=0.3, alpha=0.4, color=color, rasterized=True)
    # 30-day rolling mean
    rolling = ts_wide[pass_name].rolling(30, center=True, min_periods=10).mean()
    ax.plot(rolling.index, rolling.values, color="k", lw=0.8, label="30-day mean")
    ax.set_ylabel(f"VOD ({pass_name})")
    ax.set_ylim(0, None)
    ax.legend(loc="upper right", fontsize=8)

axes[0].set_title(f"Daily VOD at {SITE_ID} (2002–2022)")
axes[1].set_xlabel("Date")
fig.tight_layout()
fig.savefig(OUT_DIR / "amsr_vod_timeseries.png", dpi=150)
plt.show()

## 3) Seasonal climatology at US-CDM

Compute seasonal mean/stddev from the daily series above using the same season
definitions as `map.data.amsr_extract` (matching `call_ee.py` conventions).
Then compare to the values stored in the pre-computed climatology parquet.

In [ ]:
from map.data.amsr_extract import _SEASONS, _doy_in_season

# Compute seasonal stats from daily series
ts_doy = ts.copy()
ts_doy["doy"] = pd.DatetimeIndex(ts_doy["date"]).dayofyear

clim_rows = []
for pass_name in ["asc", "desc"]:
    sub = ts_doy[ts_doy["pass"] == pass_name]
    doys = sub["doy"].values
    vods = sub["vod"].values

    for season_name, (s_start, s_end) in _SEASONS.items():
        mask = _doy_in_season(doys, s_start, s_end)
        sv = vods[mask]
        sv = sv[~np.isnan(sv)]
        clim_rows.append(
            {
                "pass": pass_name,
                "season": season_name,
                "mean": np.nanmean(sv) if len(sv) else np.nan,
                "stddev": np.nanstd(sv) if len(sv) else np.nan,
                "n": len(sv),
            }
        )

    # Annual
    valid = vods[~np.isnan(vods)]
    clim_rows.append(
        {
            "pass": pass_name,
            "season": "annual",
            "mean": np.nanmean(valid) if len(valid) else np.nan,
            "stddev": np.nanstd(valid) if len(valid) else np.nan,
            "n": len(valid),
        }
    )

clim_df = pd.DataFrame(clim_rows)
print("Computed from daily series:")
print(clim_df.to_string(index=False))

# Compare to pre-computed parquet
if os.path.exists(VOD_CLIM_PATH):
    pq = pd.read_parquet(VOD_CLIM_PATH)
    cdm_row = pq[pq["sample_id"].str.startswith(f"reesh_{SITE_ID}")]
    if len(cdm_row):
        print(f"\nPre-computed parquet ({SITE_ID}):")
        vod_cols = [c for c in cdm_row.columns if c.startswith("vod_")]
        print(cdm_row[vod_cols].T.to_string())
    else:
        print(f"\n{SITE_ID} not found in climatology parquet")
else:
    print(f"\nClimatology parquet not found: {VOD_CLIM_PATH}")

# Bar chart: seasonal mean VOD by pass
season_order = ["winter", "spring", "summer", "autumn", "annual"]
clim_df["season"] = pd.Categorical(
    clim_df["season"], categories=season_order, ordered=True
)
clim_df = clim_df.sort_values(["season", "pass"])

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(season_order))
w = 0.35

for i, (pass_name, color) in enumerate([("asc", "#2196F3"), ("desc", "#FF9800")]):
    sub = clim_df[clim_df["pass"] == pass_name].set_index("season").loc[season_order]
    ax.bar(
        x + i * w,
        sub["mean"],
        w,
        yerr=sub["stddev"],
        label=pass_name,
        color=color,
        alpha=0.8,
        capsize=3,
    )

ax.set_xticks(x + w / 2)
ax.set_xticklabels(season_order)
ax.set_ylabel("VOD (mean ± stddev)")
ax.set_title(f"Seasonal VOD climatology at {SITE_ID}")
ax.legend()
fig.tight_layout()
plt.show()

## 4) Climatology parquet summary (all sites)

Load the pre-computed `amsr_vod_climatology.parquet` produced by
`map.data.amsr_extract.extract_amsr_vod_climatology()`.  
Each row is one training site with 20 VOD statistic columns.

In [ ]:
assert os.path.exists(VOD_CLIM_PATH), f"Missing: {VOD_CLIM_PATH}"

vod_clim = pd.read_parquet(VOD_CLIM_PATH)
print(f"Shape: {vod_clim.shape}")
print(f"Columns: {list(vod_clim.columns)}")
print()

# NaN rates per column
vod_cols = [c for c in vod_clim.columns if c.startswith("vod_")]
nan_rates = vod_clim[vod_cols].isna().mean()
print("NaN fraction per column:")
print(nan_rates.to_string())
print()

# Descriptive stats
print(vod_clim[vod_cols].describe().round(4).to_string())

## 5) VOD distributions across training sites

Histograms of 4 representative climatology columns across all sites.

In [ ]:
plot_cols = [
    "vod_asc_mean_summer",
    "vod_desc_mean_summer",
    "vod_asc_stddev_summer",
    "vod_asc_mean_winter",
]

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, col in zip(axes.flat, plot_cols):
    vals = vod_clim[col].dropna()
    ax.hist(vals, bins=60, color="#607D8B", edgecolor="white", linewidth=0.3)
    ax.set_xlabel(col)
    ax.set_ylabel("Sites")
    ax.set_title(f"{col}  (n={len(vals)})")

fig.suptitle("VOD climatology distributions across training sites", fontsize=13)
fig.tight_layout()
fig.savefig(OUT_DIR / "amsr_vod_distributions.png", dpi=150)
plt.show()

## 6) Spatial pattern of summer VOD

Merge the climatology with site lat/lon from the training parquet and scatter-plot
`vod_asc_mean_summer` on a CONUS map.

In [ ]:
# Merge climatology with site coordinates
site_coords = sites_df[["sample_id", "lat", "lon"]].drop_duplicates(subset="sample_id")
merged = vod_clim.merge(site_coords, on="sample_id", how="inner")
print(f"Merged sites with coordinates: {len(merged)}")

# Filter to CONUS extent
conus = merged[
    (merged["lon"] >= -125)
    & (merged["lon"] <= -66)
    & (merged["lat"] >= 24)
    & (merged["lat"] <= 50)
].copy()
print(f"CONUS sites: {len(conus)}")

col = "vod_asc_mean_summer"
valid = conus[conus[col].notna()]

fig, ax = plt.subplots(figsize=(12, 6))
sc = ax.scatter(
    valid["lon"],
    valid["lat"],
    c=valid[col],
    cmap="YlGn",
    s=4,
    alpha=0.7,
    vmin=valid[col].quantile(0.02),
    vmax=valid[col].quantile(0.98),
    rasterized=True,
)
cbar = fig.colorbar(sc, ax=ax, shrink=0.7, label=col)
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Summer ascending VOD across CONUS training sites (n={len(valid)})")
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(OUT_DIR / "amsr_vod_spatial.png", dpi=150)
plt.show()